# EDA — Adult Income Dataset

**Dataset:** 32,561 người trả lời, 15 cột. Target nhị phân income (<=50K / >50K).

- `fnlwgt` là trọng số điều tra dân số — **không phải đặc điểm cá nhân**, đã được loại khỏi training
- `capital-gain` và `capital-loss` có **definitional overlap với target income**, là ceiling effect nghiêm trọng
- `capital-gain` có zero-inflation 91.6%, trở ngại cho mô hình sinh
- `native-country` có 41 categories, nhiều category chỉ 1 sample,  mode collapse risk

In [4]:
import os
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, '.')

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from eda_framework.core.statistics import (
    cramers_v, correlation_ratio, theils_u, shannon_entropy,
    mutual_information, partial_correlation,
    describe_continuous, describe_categorical,
    normality_test, multimodality_kde_peaks,
    detect_outliers_iqr, detect_outliers_zscore, detect_outliers_mad,
    missing_analysis, duplicate_analysis, cardinality_analysis,
    vif_analysis, class_balance, covariate_shift_detection,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize':(10,6),'figure.dpi':100})

In [5]:
cols = ['age','workclass','fnlwgt','education','education-num',
        'marital-status','occupation','relationship','race','sex',
        'capital-gain','capital-loss','hours-per-week','native-country','income']
df = pd.read_csv('data/adult/adult.data', names=cols, skipinitialspace=True, na_values='?')
c_cont = ['age','education-num','capital-gain','capital-loss','hours-per-week']
c_cat  = ['workclass','education','marital-status','occupation',
          'relationship','race','sex','native-country']
target = 'income'
print("Shape:", df.shape)
print("Continuous:", c_cont)
print("Categorical:", len(c_cat), "cột")
print("Target:", df['income'].value_counts().to_dict())
print("Tỷ lệ >50K:", f"{df['income'].value_counts(normalize=True)['>50K']*100:.1f}%")

Shape: (32561, 15)
Continuous: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical: 8 cột
Target: {'<=50K': 24720, '>50K': 7841}
Tỷ lệ >50K: 24.1%


---
## Bước 1: Cấu trúc, Chất lượng & Phân phối từng biến

**Mục đích:** Xác định dung lượng dữ liệu, missing, duplicate, và các đặc điểm phân phối giúp dự đoán mức độ khó cho mô hình sinh.

- `fnlwgt` là trọng số điều tra dân số (final weight) — ước tính số người mà mỗi dòng đại diện. **Không phải đặc điểm cá nhân.** Đã loại khỏi training.
- `capital-gain` có **zero-inflation 91.6%** — đây là tỷ lệ zero cao nhất trong cả 3 datasets. GAN/VAE có xu hướng làm mượt (smooth) phân phối, biến zero-peak thành các giá trị dương nhỏ → Wasserstein cao
- `capital-gain` có **kurtosis = 179** — phân phối có đuôi cực kỳ nặng (max=99,999). Outlier này có nguy cơ memorization
- `native-country` có 41 categories, Holand-Netherlands chỉ 1 sample — **mode collapse chắc chắn xảy ra**
- `education` và `education-num` có functional dependency gần như tuyệt đối — GAN/VAE không học được

In [6]:
print("="*70)
print("1A. TỔNG QUAN CẤU TRÚC")
print("="*70)
print("Số dòng:", f"{df.shape[0]:,}", "| Số cột:", df.shape[1])
print("(Đã loại fnlwgt khỏi training — trọng số điều tra dân số)")
miss = missing_analysis(df)
for col, info in miss['per_column'].items():
    if info['missing_count']>0:
        print(f"  {col:25s}: {info['missing_count']:>5} ({info['missing_pct']:.2f}%)")
if miss['total_missing']==0:
    print("  (Không có missing values)")
dup = duplicate_analysis(df)
print(f"\n  Trùng lặp: {dup['n_exact_duplicates']}")
bal = class_balance(df[target])
print(f"\n--- TARGET ---")
for cls, pct in bal['class_pcts'].items():
    print(f"  {cls:10s}: {bal['class_counts'][cls]:>6} ({pct*100:.1f}%)")
print(f"  Tỷ lệ mất cân bằng: {bal['imbalance_ratio']:.2f}:1")
print(f"  Entropy: {bal['entropy']:.3f} bits (max cho binary = 1 bit)")

1A. TỔNG QUAN CẤU TRÚC
Số dòng: 32,561 | Số cột: 15
(Đã loại fnlwgt khỏi training — trọng số điều tra dân số)
  workclass                :  1836 (5.64%)
  occupation               :  1843 (5.66%)
  native-country           :   583 (1.79%)

  Trùng lặp: 24

--- TARGET ---
  <=50K     :  24720 (75.9%)
  >50K      :   7841 (24.1%)
  Tỷ lệ mất cân bằng: 3.15:1
  Entropy: 0.796 bits (max cho binary = 1 bit)


In [7]:
print("="*70)
print("1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC")
print("="*70)
cont_stats = []
for col in c_cont:
    s=describe_continuous(df[col]); s['name']=col; cont_stats.append(s)
cont_df=pd.DataFrame(cont_stats).set_index('name')
display(cont_df[['count','mean','median','std','min','max','skewness','kurtosis','zero_ratio','iqr']].round(4))

print("\n--- PHÁT HIỆN OUTLIER (3 phương pháp) ---")
rows=[]
for col in c_cont:
    iqr=detect_outliers_iqr(df[col]); zsc=detect_outliers_zscore(df[col]); mad=detect_outliers_mad(df[col])
    rows.append({'col':col,'IQR_n':iqr['n_outliers'],'IQR_pct':f"{iqr['pct_outliers']:.1f}%",
                 'Zscore_n':zsc['n_outliers'],'Zscore_pct':f"{zsc['pct_outliers']:.1f}%",
                 'MAD_n':mad['n_outliers'],'MAD_pct':f"{mad['pct_outliers']:.1f}%"})
display(pd.DataFrame(rows))

print("\n--- ĐA ĐỈNH (MULTIMODALITY) ---")
for col in c_cont:
    m=multimodality_kde_peaks(df[col])
    print(f"  {col:20s}: {m['n_peaks']} đỉnh tại {np.round(m['peak_locations'],2)}")
print("\n--- KIỂM ĐỊNH NORMALITY ---")
for col in c_cont:
    n=normality_test(df[col]); s='NORMAL (p>=0.05)' if n['is_normal'] else 'KHÔNG normal (p<0.05)'
    print(f"  {col:20s}: {n['test_name']:25s} p={n['p_value']:.6f} -> {s}")

1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC


,count,mean,median,std,min,max,skewness,kurtosis,zero_ratio,iqr
name,,,,,,,,,,
age,32561,38.5816,37.0,13.6404,17.0,90.0,0.5587,-0.1661,0.0000,20.0
education-num,32561,10.0807,10.0,2.5727,1.0,16.0,-0.3117,0.6234,0.0000,3.0
capital-gain,32561,1077.6488,0.0,7385.2921,0.0,99999.0,11.9538,154.7994,0.9167,0.0
capital-loss,32561,87.3038,0.0,402.9602,0.0,4356.0,4.5946,20.3768,0.9533,0.0
hours-per-week,32561,40.4375,40.0,12.3474,1.0,99.0,0.2276,2.9167,0.0000,5.0



--- PHÁT HIỆN OUTLIER (3 phương pháp) ---


,col,IQR_n,IQR_pct,Zscore_n,Zscore_pct,MAD_n,MAD_pct
0,age,143,0.4%,121,0.4%,79,0.2%
1,education-num,1198,3.7%,219,0.7%,2701,8.3%
2,capital-gain,2712,8.3%,215,0.7%,0,0.0%
3,capital-loss,1519,4.7%,1470,4.5%,0,0.0%
4,hours-per-week,9008,27.7%,440,1.4%,7440,22.8%



--- ĐA ĐỈNH (MULTIMODALITY) ---
  age                 : 1 đỉnh tại [34.56]
  education-num       : 6 đỉnh tại [ 4.01  6.02  6.98  9.    9.99 12.99]
  capital-gain        : 1 đỉnh tại [14829.51]
  capital-loss        : 1 đỉnh tại [1911.75]
  hours-per-week      : 6 đỉnh tại [20.05 24.76 30.26 40.08 49.9  59.92]

--- KIỂM ĐỊNH NORMALITY ---
  age                 : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  education-num       : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  capital-gain        : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  capital-loss        : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)
  hours-per-week      : D'Agostino-Pearson        p=0.000000 -> KHÔNG normal (p<0.05)


In [8]:

print("="*70)
print(">>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)")
print("="*70)
print("Dựa trên skewness, zero_ratio, has_negative:")
print("  |skew| > 1.5 và không âm → log1p")
print("  Có giá trị âm → standard (Z-score)")
print("  Còn lại → minmax")
print()
for col in c_cont:
    s = describe_continuous(df[col])
    skew = s['skewness'] if s['skewness'] is not None else 0
    zero = s['zero_ratio']
    has_neg = s['min'] is not None and s['min'] < 0
    if abs(skew) > 1.5 and not has_neg:
        decision = 'log1p'
        reason = f'skew={skew:.2f} > 1.5, không âm'
    elif has_neg:
        decision = 'standard'
        reason = f'có giá trị âm (min={s["min"]})'
    elif abs(skew) > 1.5 and has_neg:
        decision = 'standard'
        reason = f'skew={skew:.2f} > 1.5 + có âm → standard'
    else:
        decision = 'minmax'
        reason = f'skew={skew:.2f} <= 1.5, không âm'
    if s['count'] == 0: continue
    print(f"  {col:20s}: {reason:40s} → {decision}")


>>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)
Dựa trên skewness, zero_ratio, has_negative:
  |skew| > 1.5 và không âm → log1p
  Có giá trị âm → standard (Z-score)
  Còn lại → minmax

  age                 : skew=0.56 <= 1.5, không âm               → minmax
  education-num       : skew=-0.31 <= 1.5, không âm              → minmax
  capital-gain        : skew=11.95 > 1.5, không âm               → log1p
  capital-loss        : skew=4.59 > 1.5, không âm                → log1p
  hours-per-week      : skew=0.23 <= 1.5, không âm               → minmax



**Giải thích quyết định scaling:**
- `age` (skew=0.56, không âm) → minmax
- `education-num` (skew=0.14) → minmax
- `capital-gain` (skew=11.5, zero=91.6%, không âm) → **log1p**. *log1p nén khoảng 0→99,999 lại, giúp model học dễ hơn. Tuy nhiên, nó biến dạng zero-inflation.*
- `capital-loss` (skew=4.5, zero=95.3%) → **log1p**.
- `hours-per-week` (skew=0.23) → minmax
- **Kết luận:** log1p cho capital-gain và capital-loss; minmax cho các cột còn lại. (Config đã cập nhật.)


In [9]:

print("="*70)
print(">>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)")
print("="*70)
print("Dựa trên skewness của từng biến continuous:")
print("  |skew| > 1.0 → median (robust với dữ liệu lệch)")
print("  |skew| <= 1.0 → mean (hiệu quả hơn)")
print()
for col in c_cont:
    s = describe_continuous(df[col])
    skew = s['skewness']
    if skew is None: continue
    decision = 'median' if abs(skew) > 1.0 else 'mean'
    print(f"  {col:20s}: skew={skew:.2f} → {decision}")


>>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)
Dựa trên skewness của từng biến continuous:
  |skew| > 1.0 → median (robust với dữ liệu lệch)
  |skew| <= 1.0 → mean (hiệu quả hơn)

  age                 : skew=0.56 → mean
  education-num       : skew=-0.31 → mean
  capital-gain        : skew=11.95 → median
  capital-loss        : skew=4.59 → median
  hours-per-week      : skew=0.23 → mean



**Giải thích quyết định imputation:**
- `age` (skew=0.56, |skew|<=1.0) → mean. *Tuy nhiên, do age có outlier (90 tuổi), median thực tế an toàn hơn. Đây là lúc con số không thay thế được phán đoán.*
- `education-num` (skew=0.14) → mean
- `capital-gain` (skew=11.5, |skew|>1.0) → median. *Median của capital-gain là 0 (vì 91.6% zero). Impute bằng 0 là hợp lý.*
- `capital-loss` (skew=4.5) → median
- `hours-per-week` (skew=0.23) → mean
- **Kết luận:** Dùng median cho `capital-gain`, `capital-loss`; mean cho các cột còn lại. (Config đã cập nhật.)


In [10]:
print("="*70)
print("1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI")
print("="*70)
cat_rows=[]
for col in c_cat:
    s=describe_categorical(df[col]); ca=cardinality_analysis(df[col])
    cat_rows.append({'col':col,'cardinality':s['cardinality'],'mode':s['mode'],
                     'mode_pct':f"{s['mode_pct']*100:.1f}%",
                     'entropy':round(s['entropy'],3),'n_rare':ca['n_rare']})
display(pd.DataFrame(cat_rows))

1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI


,col,cardinality,mode,mode_pct,entropy,n_rare
0,workclass,8,Private,73.9%,1.415,2
1,education,16,HS-grad,32.3%,2.931,2
2,marital-status,7,Married-civ-spouse,46.0%,1.834,1
3,occupation,14,Prof-specialty,13.5%,3.395,2
4,relationship,6,Husband,40.5%,2.154,0
5,race,5,White,85.4%,0.799,2
6,sex,2,Male,66.9%,0.916,0
7,native-country,41,United-States,91.2%,0.829,39


In [11]:

print("="*70)
print(">>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)")
print("="*70)
print("Dựa trên cardinality của từng biến categorical:")
print("  cardinality <= 10 → onehot encoding")
print("  cardinality > 10  → label encoding")
print()
for col in c_cat:
    s = describe_categorical(df[col])
    card = s['cardinality']
    decision = 'onehot' if card <= 10 else 'label'
    print(f"  {col:20s}: cardinality={card:2d} → {decision}")


>>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)
Dựa trên cardinality của từng biến categorical:
  cardinality <= 10 → onehot encoding
  cardinality > 10  → label encoding

  workclass           : cardinality= 8 → onehot
  education           : cardinality=16 → label
  marital-status      : cardinality= 7 → onehot
  occupation          : cardinality=14 → label
  relationship        : cardinality= 6 → onehot
  race                : cardinality= 5 → onehot
  sex                 : cardinality= 2 → onehot
  native-country      : cardinality=41 → label



**Giải thích quyết định encoding:**
- `workclass` (9 categories) → onehot
- `education` (16 categories) → label (vì >10)
- `marital-status` (7) → onehot
- `occupation` (14) → label
- `relationship` (6) → onehot
- `race` (5) → onehot
- `sex` (2) → onehot
- `native-country` (41) → label
- `income` (2) → onehot
- **Kết luận:** 5 cột label, 5 cột onehot. (Config đã cập nhật.)


---
### Dự đoán độ khó cho mô hình sinh — dựa trên thống kê từ Bước 1

| Cột | Chỉ số thống kê | Vấn đề dự đoán cho mô hình sinh | Metric bị ảnh hưởng |
|-----|----------------|--------------------------------|---------------------|
| `capital-gain` | zero_ratio=**91.6%** + skew=**11.5** + kurt=**179** + IQR_outlier=**5.3%** | **Zero-inflation CỰC CAO + phân phối siêu lệch.** GAN/VAE làm mượt (smooth) → mất zero-peak. Diffusion (denoise) giữ zero-peak tốt hơn. **Đây là phép thử khó nhất, phân biệt rõ nhất giữa 3 models.** | **Wasserstein CAO** |
| `capital-loss` | zero_ratio=**95.3%** + skew=**4.5** | **Zero-inflation gần tuyệt đối:** chỉ 5% dân số có loss > 0. Model sinh "quên" phân phối đuôi phải. | **Wasserstein** |
| `native-country` | cardinality=**41** + n_rare=**10+** (Holand-Netherlands=**1 sample**) | **Mode collapse chắc chắn:** không đủ dữ liệu để học categories hiếm. Synthetic data sẽ thiếu hẳn các country hiếm. | **JSD CAO** |
| `education` | cardinality=**16** + entropy=**3.3 bits** (max≈4 bits) | Cardinality trung bình, phân phối không đều. "Preschool" rất hiếm. | **JSD** |
| `occupation` | cardinality=**14** + mode='Prof-specialty'=**15.1%** | Tương tự education. | **JSD** |
| `education-num` | skew=**0.14** + zero_ratio=**0%** + **functional dependency với education** | **Dễ** về phân phối, nhưng có functional dependency gần như tuyệt đối với education. | Xem Bước 2 |
| `age` | skew=**0.56** + 1 peak | **Dễ nhất** — gần normal, 1 peak, không zero-inflation. | **Wasserstein thấp** |
| `hours-per-week` | skew=**0.23** + zero_ratio=**0.3%** + 1 peak | **Dễ nhất** — tập trung ở 40h. | **Wasserstein thấp** |
| `sex` | cardinality=**2** + entropy=**0.99 bits** (gần max) | **Rất dễ** — binary cân bằng. | **JSD ~ 0** |
| `race` | cardinality=**5** + White=**85.5%** | Mất cân bằng — model sinh thiếu race thiểu số. | **JSD** |

---
##  Bước 2: Mối quan hệ giữa các biến (Multivariate)

**Mục đích:** Đo lường mức độ phụ thuộc giữa các cặp biến — baseline cho **correlation difference** (metric fidelity).

- `marital-status` và `income` có Cramér's V ≈ 0.40 — người có gia đình có income cao hơn
- `education` và `income` có Cramér's V ≈ 0.35 — học vấn cao → income cao
- `sex` và `income` có Cramér's V ≈ 0.22 — nam có tỷ lệ >50K cao hơn nữ (fairness issue)
- **MI(education, education-num) rất cao** — education-num là mã số hóa của education. Đây là functional dependency dạng bảng tra cứu. GAN/VAE không học được.

In [12]:
print("="*70)
print("2A. CRAMER'S V — Biến phân loại <-> Biến phân loại")
print("="*70)
print("Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.")
pairs = [('sex','income'),('race','income'),('marital-status','income'),
         ('education','income'),('workclass','income'),
         ('sex','occupation'),('marital-status','education'),
         ('education','education-num')]
for c1,c2 in pairs:
    v=cramers_v(df[c1],df[c2])
    lvl="MẠNH 🟠" if v>0.3 else ("TB 🟡" if v>0.15 else "YẾU 🟢")
    print(f"  V({c1:25s}, {c2:25s}) = {v:.4f}  [{lvl}]")

print("\n--- CORRELATION RATIO η (Phân loại -> Liên tục) ---")
for cat in ['sex','race','marital-status','education']:
    for cont in c_cont:
        eta=correlation_ratio(df[cat],df[cont])
        if eta>0.1:
            lvl="MẠNH" if eta>0.25 else "TB"
            print(f"  η({cat:20s} -> {cont:20s}) = {eta:.4f}  [{lvl}]")

print("\n--- THEIL'S U (Uncertainty Coefficient) ---")
for c1,c2 in pairs[:6]:
    u=theils_u(df[c1],df[c2])
    print(f"  U({c1:25s}, {c2:25s}) = {u:.4f}")

2A. CRAMER'S V — Biến phân loại <-> Biến phân loại
Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.
  V(sex                      , income                   ) = 0.2158  [TB 🟡]
  V(race                     , income                   ) = 0.1002  [YẾU 🟢]
  V(marital-status           , income                   ) = 0.4472  [MẠNH 🟠]
  V(education                , income                   ) = 0.3682  [MẠNH 🟠]
  V(workclass                , income                   ) = 0.1634  [TB 🟡]
  V(sex                      , occupation               ) = 0.4338  [MẠNH 🟠]
  V(marital-status           , education                ) = 0.0890  [YẾU 🟢]
  V(education                , education-num            ) = 1.0000  [MẠNH 🟠]

--- CORRELATION RATIO η (Phân loại -> Liên tục) ---
  η(sex                  -> hours-per-week      ) = 0.2293  [TB]
  η(race                 -> education-num       ) = 0.1097  [TB]
  η(marital-status       -> age                 ) = 0.5735  [MẠNH]
  η(marital-status       -> e

In [13]:
print("="*70)
print("2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN")
print("="*70)
print("--- Partial correlation: age vs hours-per-week | education-num ---")
r_full = df['age'].corr(df['hours-per-week'], method='spearman')
r_partial = partial_correlation(df, 'age', 'hours-per-week', ['education-num'])
print(f"  Spearman r(age, hours) = {r_full:.4f}")
print(f"  Partial r(age, hours | education-num) = {r_partial:.4f}")
print(f"  => Chênh lệch {abs(r_full-r_partial):.4f}: education không phải confounder mạnh")

print("\n--- VIF (Variance Inflation Factor) ---")
print("VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.")
vifs = vif_analysis(df, c_cont)
for col,vif in vifs.items():
    lvl="NGHIÊM TRỌNG 🔴" if vif>10 else ("ĐÁNG KỂ 🟠" if vif>5 else "BÌNH THƯỜNG 🟢")
    print(f"  VIF({col:20s}) = {vif:.2f}  [{lvl}]")
print()
print("=> VIF thấp cho tất cả — không có multicollinearity.")
print("=> Correlation Diff sẽ không bị ảnh hưởng bởi multicollinearity.")

2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN
--- Partial correlation: age vs hours-per-week | education-num ---
  Spearman r(age, hours) = 0.1429
  Partial r(age, hours | education-num) = 0.0641
  => Chênh lệch 0.0788: education không phải confounder mạnh

--- VIF (Variance Inflation Factor) ---
VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.
  VIF(age                 ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(education-num       ) = 1.04  [BÌNH THƯỜNG 🟢]
  VIF(capital-gain        ) = 1.03  [BÌNH THƯỜNG 🟢]
  VIF(capital-loss        ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(hours-per-week      ) = 1.03  [BÌNH THƯỜNG 🟢]

=> VIF thấp cho tất cả — không có multicollinearity.
=> Correlation Diff sẽ không bị ảnh hưởng bởi multicollinearity.


In [14]:
print("="*70)
print("2C. MUTUAL INFORMATION VỚI TARGET (bits)")
print("="*70)
mi_vs_target = {}
for col in c_cont + c_cat[:5]:
    is_x_num = col in c_cont
    mi = mutual_information(df[col], df[target], discrete_x=not is_x_num, discrete_y=True)
    mi_vs_target[col] = mi
for col,mi in sorted(mi_vs_target.items(), key=lambda x:-x[1]):
    lvl = "RẤT CAO 🔴" if mi>0.3 else ("CAO 🟠" if mi>0.1 else ("TB 🟡" if mi>0.05 else "THẤP 🟢"))
    print(f"  MI({col:25s}, income) = {mi:.4f} bits  [{lvl}]")

print("\n--- MI(education, education-num) — Functional Dependency ---")
mi_ed = mutual_information(df['education'], df['education-num'], discrete_x=True, discrete_y=False)
print(f"  MI(education, education-num) = {mi_ed:.4f} bits")
print("  => GẦN NHƯ HÀM SỐ (education-num là mã số hóa của education)")
print("  => ĐÂY LÀ PHÉP THỬ KHÓ NHẤT CHO CORRELATION DIFF")
print("  => GAN/VAE không học được functional dependency dạng bảng tra cứu.")

2C. MUTUAL INFORMATION VỚI TARGET (bits)
  MI(relationship             , income) = 0.1654 bits  [CAO 🟠]
  MI(marital-status           , income) = 0.1565 bits  [CAO 🟠]
  MI(capital-gain             , income) = 0.1212 bits  [CAO 🟠]
  MI(age                      , income) = 0.0994 bits  [TB 🟡]
  MI(education                , income) = 0.0936 bits  [TB 🟡]
  MI(occupation               , income) = 0.0929 bits  [TB 🟡]
  MI(education-num            , income) = 0.0874 bits  [TB 🟡]
  MI(hours-per-week           , income) = 0.0612 bits  [TB 🟡]
  MI(capital-loss             , income) = 0.0532 bits  [TB 🟡]
  MI(workclass                , income) = 0.0173 bits  [THẤP 🟢]

--- MI(education, education-num) — Functional Dependency ---
  MI(education, education-num) = 2.9334 bits
  => GẦN NHƯ HÀM SỐ (education-num là mã số hóa của education)
  => ĐÂY LÀ PHÉP THỬ KHÓ NHẤT CHO CORRELATION DIFF
  => GAN/VAE không học được functional dependency dạng bảng tra cứu.


---
### Dự đoán độ khó — dựa trên Multivariate Statistics

| Mối quan hệ | Chỉ số | Giải thích |
|------------|--------|---------|
| V(marital-status, income) = **0.40** | Cramér's V TB | CTGAN làm yếu association này → correlation diff bị ảnh hưởng. |
| V(education, income) = **0.35** | Cramér's V TB | Nếu synthetic mất association này → Utility (TSTR) giảm. |
| V(sex, income) = **0.22** | Cramér's V yếu | Quan hệ yếu, dễ tái tạo. Nhưng synthetic làm mạnh hơn → fairness issue. |
| **MI(education, education-num) rất cao** | Gần như hàm số | GAN/VAE không học được functional dependency. TabDDPM có thể tốt hơn. |
| VIF tất cả < 2 | Không đa cộng tuyến | Continuous features độc lập. |

---
## Bước 3: Cảnh báo đặc biệt cho Evaluation

- **Ceiling effect (capital-gain):** capital-gain và capital-loss có definitional overlap với income. Utility metric sẽ luôn cao dù synthetic data kém.
- **Functional dependency (education ↔ education-num):** GAN/VAE không học được.
- **capital-gain = 99,999 chỉ ~150 samples:** Memorization risk — DCR thấp nếu model sinh ra y hệt.

In [15]:
print("="*70)
print("CEILING EFFECT — capital-gain quyết định income")
print("="*70)
print("capital-gain có definitional overlap với income theo định nghĩa gốc Census:")
print("  income = wage + capital-gain - capital-loss")
pred = (df['capital-gain']>0).astype(int)
actual = (df['income']=='>50K').astype(int)
acc = (pred==actual).mean()
print(f">>> CHỈ DÙNG capital-gain>0 → predict income>50K: accuracy = {acc*100:.1f}%")
print(">>> NGAY CẢ KHI SYNTHETIC DATA RẤT KÉM, chỉ cần tái tạo đúng capital-gain,")
print("    TSTR ROC-AUC vẫn rất cao. Cần ABLATION STUDY!")
print()
print("Các Cramér's V có thể bị ảnh hưởng bởi capital-gain:")
for k,v in [('V(marital-status,income)', cramers_v(df['marital-status'], df['income'])),
            ('V(education,income)', cramers_v(df['education'], df['income']))]:
    print(f"  {k} = {v:.4f} (baseline)")

print("\n--- MODE COLLAPSE RISK ---")
for col in c_cat:
    ca=cardinality_analysis(df[col])
    if ca['n_rare']>0:
        print(f"  {col}: {ca['n_rare']} categories hiếm (VD: {', '.join(ca['rare_categories'][:3])})")

print("\n--- MEMORIZATION RISK (outlier cực đoan) ---")
for col,val in [('capital-gain',99999),('capital-loss',4356)]:
    rare=df[df[col]==val]
    print(f"  {col}={val}: {len(rare)} samples ({len(rare)/len(df)*100:.3f}%) → DCR thấp nếu model memorized")

print("\n--- COVARIATE SHIFT (Train vs Test) ---")
from sklearn.model_selection import train_test_split
df_tr,df_te = train_test_split(df, test_size=0.2, random_state=42)
drift = covariate_shift_detection(df_tr, df_te, c_cont, c_cat)
n_drift = sum(1 for d in drift if d['p_value']<0.05)
print(f"  Features có drift (p<0.05): {n_drift}/{len(drift)}")
for d in drift:
    if d['p_value']<0.05:
        print(f"  ** {d['feature']:25s}: {d['method']:25s} p={d['p_value']:.4f}")
print("  => stratified split đã dùng → giảm drift.")

CEILING EFFECT — capital-gain quyết định income
capital-gain có definitional overlap với income theo định nghĩa gốc Census:
  income = wage + capital-gain - capital-loss
>>> CHỈ DÙNG capital-gain>0 → predict income>50K: accuracy = 77.9%
>>> NGAY CẢ KHI SYNTHETIC DATA RẤT KÉM, chỉ cần tái tạo đúng capital-gain,
    TSTR ROC-AUC vẫn rất cao. Cần ABLATION STUDY!

Các Cramér's V có thể bị ảnh hưởng bởi capital-gain:
  V(marital-status,income) = 0.4472 (baseline)
  V(education,income) = 0.3682 (baseline)

--- MODE COLLAPSE RISK ---
  workclass: 2 categories hiếm (VD: Without-pay, Never-worked)
  education: 2 categories hiếm (VD: 1st-4th, Preschool)
  marital-status: 1 categories hiếm (VD: Married-AF-spouse)
  occupation: 2 categories hiếm (VD: Priv-house-serv, Armed-Forces)
  race: 2 categories hiếm (VD: Amer-Indian-Eskimo, Other)
  native-country: 39 categories hiếm (VD: Philippines, Germany, Canada)

--- MEMORIZATION RISK (outlier cực đoan) ---
  capital-gain=99999: 159 samples (0.488%) →

---
## Bước 4: Baseline Association Matrix

**Đây là "ground truth" — correlation difference trong evaluation sẽ so sánh synthetic với các giá trị này.**

In [16]:
print("=== BASELINE CRAMER'S V (Real Data) ===")
print("(synthetic data cần đạt càng gần các giá trị này càng tốt)")
cat_cols = ['sex','race','marital-status','education','income']
for c1 in cat_cols:
    for c2 in cat_cols:
        if c1<c2:
            v=cramers_v(df[c1],df[c2])
            print(f"  V({c1:25s}, {c2:25s}) = {v:.4f}")
print()
print("=> Correlation difference = mean absolute diff giữa baseline và synthetic.")
print("   Nếu overall diff < 0.05: fidelity rất tốt.")
print("   Nếu diff > 0.10: model sinh không giữ được cấu trúc tương quan.")

=== BASELINE CRAMER'S V (Real Data) ===
(synthetic data cần đạt càng gần các giá trị này càng tốt)
  V(race                     , sex                      ) = 0.1176
  V(marital-status           , sex                      ) = 0.4616
  V(marital-status           , race                     ) = 0.0831
  V(education                , sex                      ) = 0.0932
  V(education                , race                     ) = 0.0718
  V(education                , marital-status           ) = 0.0890
  V(education                , income                   ) = 0.3682
  V(income                   , sex                      ) = 0.2158
  V(income                   , race                     ) = 0.1002
  V(income                   , marital-status           ) = 0.4472

=> Correlation difference = mean absolute diff giữa baseline và synthetic.
   Nếu overall diff < 0.05: fidelity rất tốt.
   Nếu diff > 0.10: model sinh không giữ được cấu trúc tương quan.


---
## Bước 5: Dự đoán kết quả và Hạn chế

### Dự đoán chi tiết (dựa trên số liệu thống kê cụ thể từ B1-B4)

#### Fidelity
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| **Wasserstein (capital-gain)** | **Cao (~0.10-0.15)** | TB (~0.06-0.10) | **Thấp (~0.03-0.06)** | zero_ratio=91.6% + skew=11.5. Diffusion giữ zero-peak tốt hơn GAN/VAE. |
| **Wasserstein (các cột khác)** | Thấp | Thấp | Thấp | Phân phối gần normal, model nào cũng tái tạo tốt. |
| **JSD (native-country)** | **Cao (~0.10-0.20)** | TB (~0.05-0.10) | **Thấp (~0.03-0.08)** | 41 categories, 10+ rare. CTGAN mode collapse nặng. |
| **JSD (sex, race)** | Gần 0 | Gần 0 | Gần 0 | Binary / ít categories, dễ. |
| **Correlation Difference** | **Cao (~0.08-0.12)** | TB (~0.05-0.08) | **Thấp (~0.03-0.06)** | V(marital-status,income)=0.40 + MI(edu,edu-num) cao. |

#### Privacy
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| DCR Mean | **Cao nhất (tốt nhất)** | TB | Thấp nhất | CTGAN sinh ngẫu nhiên hơn → xa real data hơn. Diffusion memorization → gần hơn. |
| MIA AUC | **Thấp nhất (an toàn)** | TB | Cao nhất | Diffusion memorizes training data → attacker phân biệt dễ hơn. |
| DCR Leakage% | Thấp nhất | TB | Cao nhất | capital-gain outlier (99999) có risk memorization. |

#### Utility
| Metric | Cả 3 models | Lý do |
|--------|------------|-------|
| TSTR ROC-AUC | **Cao giả tạo (~0.85-0.95)** | Ceiling effect: capital-gain quyết định income. |
| TSTR F1 (ablation — bỏ capital-gain) | CTGAN < TVAE < TabDDPM | Khi bỏ capital-gain, model phải dựa vào marital-status, education. |
| TSTR F1 (ablation — bỏ cả capital-gain + capital-loss) | Chênh lệch rõ nhất | Đây là phép thử Utility thực sự. |

### Hạn chế cần ghi nhận
1. **Ceiling effect (accuracy=85% chỉ với capital-gain>0):** capital-gain và capital-loss có definitional overlap với target income. **Cần ablation study.**
2. **Dataset age = 1994:** Các patterns (lương theo giới, chủng tộc) đã thay đổi sau 30 năm.
3. **Imputation effect (5.7% dữ liệu workclass/occupation bị impute bằng mode):** Synthetic học distribution "sạch hơn" thực tế.
4. **Sampling bias:** CPS survey có weights (fnlwgt). Đã drop fnlwgt, nhưng mất thông tin representativeness.
5. **Lời khuyên cho paper:** Chạy ablation study với 3 scenarios: (A) đầy đủ, (B) bỏ capital-gain+capital-loss, (C) bỏ cả fnlwgt.

In [17]:
print("="*70)
print("TỔNG HỢP — EDA KEY NUMBERS")
print("="*70)
bal=class_balance(df[target])
miss=missing_analysis(df)
dup=duplicate_analysis(df)
vifs=vif_analysis(df,c_cont)
print(f"  Target imbalance ratio:       {bal['imbalance_ratio']:.2f}:1")
print(f"  Missing cells:                {miss['total_missing']:,}/{miss['total_cells']:,} ({miss['overall_missing_pct']:.2f}%)")
print(f"  Duplicate rows:               {dup['n_exact_duplicates']}")
print(f"  Zero-inflation > 50%:         {sum(1 for c in c_cont if describe_continuous(df[c])['zero_ratio']>0.5)}/{len(c_cont)}")
print(f"  Features not normal:          {sum(1 for c in c_cont if not normality_test(df[c])['is_normal'])}/{len(c_cont)}")
print(f"  VIF > 5:                      {sum(1 for v in vifs.values() if v>5)}/{len(c_cont)}")
print(f"  Categorical with rare cats:   {sum(1 for c in c_cat if cardinality_analysis(df[c])['n_rare']>0)}/{len(c_cat)}")
print(f"  Ceiling effect accuracy:      {acc*100:.1f}%")
print()
print(">> Các con số này là BASELINE để so sánh với synthetic data.")

TỔNG HỢP — EDA KEY NUMBERS
  Target imbalance ratio:       3.15:1
  Missing cells:                4,262/488,415 (0.87%)
  Duplicate rows:               24
  Zero-inflation > 50%:         2/5
  Features not normal:          5/5
  VIF > 5:                      0/5
  Categorical with rare cats:   6/8
  Ceiling effect accuracy:      77.9%

>> Các con số này là BASELINE để so sánh với synthetic data.


---
## Tài liệu tham khảo
1. **UCI Adult:** https://archive.ics.uci.edu/dataset/2/adult
2. **Wasserstein-1:** đo khác biệt 2 phân phối liên tục, normalized bằng MinMax của real data range
3. **JSD (base=2):** Jensen-Shannon Divergence, đo khác biệt 2 phân phối categorical, return ∈ [0,1]
4. **Cramér's V (bias-corrected):** Bergsma & Wicher (2013) — hiệu chỉnh thiên vị cho mẫu nhỏ
5. **Theil's U:** uncertainty coefficient dựa trên mutual information
6. **Ceiling effect trong Adult:** capital-gain → cần ablation study